In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

import sys
sys.path.append("../../utils/")

from utils import *

/home/javier/.conda/envs/tfgClean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===== RUTAS =====
PROJECT_ROOT = Path.cwd().resolve().parents[2]

NOMBRE_DATASET_RESULTADO = "CIC17__cleanning__v1"

# Ajusta esto si hace falta
RUTA_BASE_RAW = PROJECT_ROOT / "02_datasets" / "raw" / "CIC17"
RUTA_SALIDA = PROJECT_ROOT / "02_datasets" / "processed" / NOMBRE_DATASET_RESULTADO

NOMBRE_DATASET_LIMPIO = f"{NOMBRE_DATASET_RESULTADO}.csv"

# ===== PARÁMETROS =====
LABEL_COL = "LABEL"
CORR_THRESHOLD = 0.95
MIN_CLASS_IMPUTE = 50

# ===== COLUMNAS A ELIMINAR MANUALMENTE SI EXISTEN =====
COLUMNAS_A_ELIMINAR = [
    "FLOW_ID",
    "SRC_IP",
    "SRC_PORT",
    "DST_IP"
]

In [3]:
print("Carpeta actual del notebook:")
print(PROJECT_ROOT)
print()
print("Ruta base raw:")
print(Path(RUTA_BASE_RAW).resolve())
print()
print("Ruta salida:")
print(Path(RUTA_SALIDA).resolve())
print()

Carpeta actual del notebook:
/home/javier/TFG_MODELOS_SIMPLES

Ruta base raw:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/raw/CIC17

Ruta salida:
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__cleanning__v1



In [4]:
df = cargar_dataset(
    ruta_base=RUTA_BASE_RAW
)

shape_original = df.shape

print("Forma original del dataset:")
print(shape_original)

df.head()

Forma original del dataset:
(2830743, 79)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [5]:
df = homogeneizar_columnas(df)

print("Primeras columnas tras homogeneización:")
print(df.columns.tolist()[:20])

Primeras columnas tras homogeneización:
['DESTINATION_PORT', 'FLOW_DURATION', 'TOTAL_FWD_PACKETS', 'TOTAL_BACKWARD_PACKETS', 'TOTAL_LENGTH_OF_FWD_PACKETS', 'TOTAL_LENGTH_OF_BWD_PACKETS', 'FWD_PACKET_LENGTH_MAX', 'FWD_PACKET_LENGTH_MIN', 'FWD_PACKET_LENGTH_MEAN', 'FWD_PACKET_LENGTH_STD', 'BWD_PACKET_LENGTH_MAX', 'BWD_PACKET_LENGTH_MIN', 'BWD_PACKET_LENGTH_MEAN', 'BWD_PACKET_LENGTH_STD', 'FLOW_BYTES_S', 'FLOW_PACKETS_S', 'FLOW_IAT_MEAN', 'FLOW_IAT_STD', 'FLOW_IAT_MAX', 'FLOW_IAT_MIN']


In [6]:
if LABEL_COL not in df.columns:
    raise ValueError(f"No se encontró la columna {LABEL_COL}")

print("Columna objetivo encontrada correctamente.")
print()
print("Distribución inicial de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Columna objetivo encontrada correctamente.

Distribución inicial de clases:


,count
LABEL,
BENIGN,2273097
DoS Hulk,231073
PortScan,158930
DDoS,128027
DoS GoldenEye,10293
FTP-Patator,7938
SSH-Patator,5897
DoS slowloris,5796
DoS Slowhttptest,5499


In [7]:
columnas_presentes_para_eliminar = [c for c in COLUMNAS_A_ELIMINAR if c in df.columns]

df = eliminar_columnas(df, columnas_presentes_para_eliminar)

print("Columnas eliminadas manualmente:")
print(columnas_presentes_para_eliminar)
print()
print("Forma actual:")
print(df.shape)

Columnas eliminadas manualmente:
[]

Forma actual:
(2830743, 79)


In [8]:
df = limpiar_infinitos_y_vacios(df)

print("Limpieza de infinitos y vacíos completada.")

Limpieza de infinitos y vacíos completada.


In [9]:
filas_antes_duplicados = len(df)

df = eliminar_filas_duplicadas(df)

filas_despues_duplicados = len(df)
duplicados_eliminados = filas_antes_duplicados - filas_despues_duplicados

print("Duplicados eliminados:", duplicados_eliminados)
print("Forma actual:", df.shape)

Duplicados eliminados: 308381
Forma actual: (2522362, 79)


In [10]:
df_sin_nulos, df_con_nulos = separar_filas_con_y_sin_nulos(df)

print("Filas sin nulos:", len(df_sin_nulos))
print("Filas con nulos:", len(df_con_nulos))

Filas sin nulos: 2520798
Filas con nulos: 1564


In [11]:
df, filas_imputadas, filas_eliminadas_nulos = imputar_o_eliminar_nulos_por_clase(
    df_sin_nulos=df_sin_nulos,
    df_con_nulos=df_con_nulos,
    label_col=LABEL_COL,
    min_class_impute=MIN_CLASS_IMPUTE
)

print("Filas imputadas:", filas_imputadas)
print("Filas eliminadas por nulos:", filas_eliminadas_nulos)
print("Forma actual:", df.shape)

Filas imputadas: 0
Filas eliminadas por nulos: 1564
Forma actual: (2520798, 79)


In [12]:
columnas_antes_constantes = df.shape[1]

df = eliminar_columnas_constantes(df)

columnas_despues_constantes = df.shape[1]
constantes_eliminadas = columnas_antes_constantes - columnas_despues_constantes

print("Columnas constantes eliminadas:", constantes_eliminadas)
print("Forma actual:", df.shape)

Columnas constantes eliminadas: 8
Forma actual: (2520798, 71)


In [13]:
df, columnas_categoricas_codificadas = codificar_columnas_categoricas_one_hot(
    df,
    label_col=LABEL_COL
)

print("Columnas categóricas originales codificadas:")
print(columnas_categoricas_codificadas)
print()
print("Forma actual:", df.shape)

Columnas categóricas originales codificadas:
[]

Forma actual: (2520798, 71)


In [14]:
columnas_antes_corr = df.shape[1]

df = eliminar_columnas_altamente_correlacionadas(
    df,
    threshold=CORR_THRESHOLD,
    label_col=LABEL_COL
)

columnas_despues_corr = df.shape[1]
corr_eliminadas = columnas_antes_corr - columnas_despues_corr

print("Columnas eliminadas por alta correlación:", corr_eliminadas)
print("Forma final tras limpieza:", df.shape)

Columnas eliminadas por alta correlación: 23
Forma final tras limpieza: (2520798, 48)


In [15]:
feature_cols = [c for c in df.columns if c != LABEL_COL]
df = df[feature_cols + [LABEL_COL]]

print("Última columna:", df.columns[-1])
df.head()

Última columna: LABEL


,DESTINATION_PORT,FLOW_DURATION,TOTAL_FWD_PACKETS,TOTAL_LENGTH_OF_FWD_PACKETS,FWD_PACKET_LENGTH_MAX,FWD_PACKET_LENGTH_MIN,FWD_PACKET_LENGTH_MEAN,BWD_PACKET_LENGTH_MAX,BWD_PACKET_LENGTH_MIN,FLOW_BYTES_S,...,INIT_WIN_BYTES_FORWARD,INIT_WIN_BYTES_BACKWARD,ACT_DATA_PKT_FWD,MIN_SEG_SIZE_FORWARD,ACTIVE_MEAN,ACTIVE_STD,ACTIVE_MAX,ACTIVE_MIN,IDLE_STD,LABEL
0,54865,3,2,12,6,6,6.0,0,0,4.000000e+06,...,33,-1,1,20,0.0,0.0,0,0,0.0,BENIGN
1,55054,109,1,6,6,6,6.0,6,6,1.100917e+05,...,29,256,0,20,0.0,0.0,0,0,0.0,BENIGN
2,55055,52,1,6,6,6,6.0,6,6,2.307692e+05,...,29,256,0,20,0.0,0.0,0,0,0.0,BENIGN
3,46236,34,1,6,6,6,6.0,6,6,3.529412e+05,...,31,329,0,20,0.0,0.0,0,0,0.0,BENIGN
4,54863,3,2,12,6,6,6.0,0,0,4.000000e+06,...,32,-1,1,20,0.0,0.0,0,0,0.0,BENIGN


In [16]:
df, mapping = codificar_etiqueta_label(df, LABEL_COL)
mapping

{'BENIGN': 0,
 'DoS Hulk': 1,
 'DDoS': 2,
 'PortScan': 3,
 'DoS GoldenEye': 4,
 'FTP-Patator': 5,
 'DoS slowloris': 6,
 'DoS Slowhttptest': 7,
 'SSH-Patator': 8,
 'Bot': 9,
 'Web Attack � Brute Force': 10,
 'Web Attack � XSS': 11,
 'Infiltration': 12,
 'Web Attack � Sql Injection': 13,
 'Heartbleed': 14}

In [17]:
print("Distribución final de clases:")
display(df[LABEL_COL].value_counts(dropna=False).to_frame("count"))

Distribución final de clases:


,count
LABEL,
0,2095057
1,172846
2,128014
3,90694
4,10286
5,5931
6,5385
7,5228
8,3219


In [18]:
guardar_dataset_csv(
    df=df,
    nombre_archivo=NOMBRE_DATASET_LIMPIO,
    ruta=RUTA_SALIDA
)

print("Dataset limpio guardado correctamente.")
print(Path(RUTA_SALIDA).resolve() / NOMBRE_DATASET_LIMPIO)

Dataset limpio guardado correctamente.
/home/javier/TFG_MODELOS_SIMPLES/02_datasets/processed/CIC17__cleanning__v1/CIC17__cleanning__v1.csv
